# Atlassian India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** www.atlassian.com/company/careers/all-jobs

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 21:32:04
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Atlassian"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Atlassian/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("ATLASSIAN INDIA JOB SCRAPER")
print("Source: www.atlassian.com/company/careers/all-jobs")
print("Method: Selenium + JSON API discovery")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


atlassian_jobs = []

# First, try Atlassian's internal JSON API (common pattern for React-based career pages)
session = get_session()
session.headers.update({"Accept": "application/json"})

# Atlassian uses a careers API - try common endpoints
API_ATTEMPTS = [
    "https://www.atlassian.com/endpoint/careers/api/jobs?location=India",
    "https://api.greenhouse.io/v1/boards/atlassian/jobs?content=true",
    "https://boards-api.greenhouse.io/v1/boards/atlassian/jobs?content=true",
]

for api_url in API_ATTEMPTS:
    try:
        print(f"  Trying API: {api_url}")
        resp = session.get(api_url, timeout=20)
        if resp.status_code == 200:
            data = resp.json()
            jobs_list = data.get("jobs", data.get("positions", data.get("results", [])))
            if jobs_list:
                india_keywords = ["india", "bengaluru", "bangalore", "pune", "hyderabad", "sydney apac"]
                for job in jobs_list:
                    loc = (job.get("location", {}).get("name", "") if isinstance(job.get("location"), dict)
                           else str(job.get("location", "")))
                    if not any(k in loc.lower() for k in india_keywords) and len(atlassian_jobs) < 3:
                        # Accept all if we have no matches yet (location may be broad)
                        pass
                    elif not any(k in loc.lower() for k in india_keywords):
                        continue

                    dept = ""
                    if isinstance(job.get("departments"), list) and job["departments"]:
                        dept = job["departments"][0].get("name", "")

                    jd_html = job.get("content", job.get("description", ""))
                    jd_text = html_to_text(jd_html)

                    job_id = str(job.get("id", job.get("job_id", len(atlassian_jobs))))
                    job_url = job.get("absolute_url", job.get("url", ""))

                    atlassian_jobs.append({
                        "job_id": job_id,
                        "title": job.get("title", job.get("name", "")),
                        "company_name": "Atlassian",
                        "job_url": job_url,
                        "business_unit": dept,
                        "raw_jd_text": jd_text,
                        "location_city": loc.split(",")[0].strip() if loc else "India",
                        "location_country": "India",
                        "industry": "Technology / DevOps / Collaboration",
                        "date_posted": str(job.get("updated_at", datetime.now().strftime("%Y-%m-%d")))[:10],
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "Atlassian API",
                    })

                print(f"  API success: {len(atlassian_jobs)} India jobs from {api_url}")
                if atlassian_jobs:
                    break
    except Exception as e:
        print(f"  API {api_url} failed: {e}")

# Selenium fallback on Atlassian's careers page
if len(atlassian_jobs) < 5:
    print("\n  Trying Selenium on atlassian.com/company/careers/all-jobs...")
    driver = setup_selenium()
    try:
        url = "https://www.atlassian.com/company/careers/all-jobs?team=&location=India"
        driver.get(url)
        time.sleep(10)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR,
                    "[class*='JobCard'], [class*='job-card'], [class*='career-listing'], a[href*='/careers/']"))
            )
        except:
            time.sleep(8)

        # Try to intercept JSON from network (Atlassian loads jobs via XHR)
        # Check window.__NEXT_DATA__ or similar React page data
        try:
            next_data = driver.execute_script("return JSON.stringify(window.__NEXT_DATA__ || {})")
            if next_data and len(next_data) > 100:
                nd = json.loads(next_data)
                # Navigate the Next.js data tree to find job listings
                def find_jobs_in_dict(d, depth=0):
                    if depth > 8:
                        return []
                    if isinstance(d, list):
                        if len(d) > 3 and all(isinstance(i, dict) and
                            ("title" in i or "jobTitle" in i or "name" in i) for i in d[:3]):
                            return d
                        for item in d:
                            result = find_jobs_in_dict(item, depth+1)
                            if result:
                                return result
                    elif isinstance(d, dict):
                        for key in ["jobs", "positions", "listings", "careers", "allJobs"]:
                            if key in d and isinstance(d[key], (list, dict)):
                                return find_jobs_in_dict(d[key], depth+1)
                        for v in d.values():
                            result = find_jobs_in_dict(v, depth+1)
                            if result:
                                return result
                    return []

                jobs_from_ssr = find_jobs_in_dict(nd)
                if jobs_from_ssr:
                    print(f"  Found {len(jobs_from_ssr)} jobs from SSR data")
                    india_keywords = ["india", "bengaluru", "bangalore", "pune", "hyderabad"]
                    for job in jobs_from_ssr:
                        loc = str(job.get("location", job.get("office", "India")))
                        if any(k in loc.lower() for k in india_keywords) or len(atlassian_jobs) == 0:
                            job_id = str(job.get("id", job.get("jobId", len(atlassian_jobs))))
                            atlassian_jobs.append({
                                "job_id": job_id,
                                "title": job.get("title", job.get("jobTitle", job.get("name", ""))),
                                "company_name": "Atlassian",
                                "job_url": job.get("url", job.get("applyUrl", "")),
                                "business_unit": str(job.get("team", job.get("department", ""))),
                                "raw_jd_text": str(job.get("description", "")),
                                "location_city": loc.split(",")[0].strip(),
                                "location_country": "India",
                                "industry": "Technology / DevOps / Collaboration",
                                "date_posted": datetime.now().strftime("%Y-%m-%d"),
                                "is_active": True,
                                "salary_currency": "INR",
                                "source_platform": "Atlassian SSR",
                            })
        except Exception as e:
            print(f"  SSR extraction failed: {e}")

        # Pure DOM scraping as last resort
        if not atlassian_jobs:
            for scroll in range(10):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "lxml")
            cards = soup.select(
                "[class*='JobCard'], [class*='job-card'], [class*='JobListing'], "
                "[class*='career-item'], li[class*='job']"
            )
            for card in cards:
                title_el = card.select_one("h3, h2, a, [class*='title']")
                title = title_el.get_text(strip=True) if title_el else ""
                href = title_el.get("href", "") if title_el and title_el.name == "a" else ""
                if not href:
                    link = card.select_one("a[href]")
                    href = link.get("href", "") if link else ""

                loc_el = card.select_one("[class*='location'], [class*='office']")
                loc = loc_el.get_text(strip=True) if loc_el else "India"

                if is_valid_job_title(title):
                    full_url = href if href.startswith("http") else f"https://www.atlassian.com{href}" if href else ""
                    atlassian_jobs.append({
                        "job_id": href.split("/")[-1] if href else str(len(atlassian_jobs)),
                        "title": title,
                        "company_name": "Atlassian",
                        "job_url": full_url,
                        "business_unit": "",
                        "raw_jd_text": card.get_text(" ", strip=True),
                        "location_city": loc.split(",")[0].strip(),
                        "location_country": "India",
                        "industry": "Technology / DevOps / Collaboration",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "Atlassian Selenium",
                    })

    except Exception as e:
        print(f"  Selenium error: {e}")
        import traceback; traceback.print_exc()
    finally:
        driver.quit()

print(f"Total Atlassian India jobs: {len(atlassian_jobs)}")


ATLASSIAN INDIA JOB SCRAPER
Source: www.atlassian.com/company/careers/all-jobs
Method: Selenium + JSON API discovery
  Trying API: https://www.atlassian.com/endpoint/careers/api/jobs?location=India


  Trying API: https://api.greenhouse.io/v1/boards/atlassian/jobs?content=true


  Trying API: https://boards-api.greenhouse.io/v1/boards/atlassian/jobs?content=true



  Trying Selenium on atlassian.com/company/careers/all-jobs...


Total Atlassian India jobs: 1


In [5]:
df_atlassian = save_results(atlassian_jobs, "Atlassian", OUTPUT_DIR)
if df_atlassian is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_atlassian.columns]
    print(df_atlassian[cols].head(10).to_string())


  [OK] Saved 1 jobs -> Atlassian_jobs_2026-03-31.csv
       Seniority: {'mid': 1}
       Work mode: {'onsite': 1}
       Has JD text: 0/1
       Has job URL: 1/1
       Has business unit: 0/1

Sample jobs:
         title location_city seniority_level business_unit                                             job_url
0  Browse Jobs         India             mid                https://www.atlassian.com/company/careers/all-jobs
